# Houston Storm Impact & Resource Planner

**Your name:** add here

This notebook contains all code and cached data. Open it in Google Colab using **File → Upload notebook**, then choose **Runtime → Run all**. No separate dataset upload or code insertion is needed. No GPU is needed. Enable your existing **CARTO_API_KEY** Secret for this notebook if you want street tiles; maps and calculations still work without it, and no key-entry prompt is used.

The first GUI adds historical power-outage, tree/debris, flood and building-report filters; source damage descriptions; a date-checked utility snapshot; and independent ambulance, repair, cooling and community/medical placement controls. It uses native outage observations and public service reports. The original people/supply GUI remains in an optional later cell.

**Scope:** red means historical recorded county outage intensity; green-yellow means tree/debris report density. No future neighborhood outage or tree-strike probability model is fitted or validated. Existing hospitals stay fixed examples; temporary posts are selected among candidate centers. Source windows, missing data, stale timestamps and unverified facility capacity are explicit.


In [ ]:
#@title 1. Load bundled files and install libraries (no data uploads needed)
import base64, io, zipfile
from pathlib import Path
payload = """..."""
project = Path('/content/houston_storm_layers')
with zipfile.ZipFile(io.BytesIO(base64.b64decode(payload))) as archive:
    for member in archive.infolist():
        target = (Path('/content') / member.filename).resolve()
        if not target.is_relative_to(project.resolve()):
            raise ValueError('Unexpected project path in bundled files')
    archive.extractall('/content')
print('All project files and cached datasets are ready. No individual data uploads are required.')
%pip -q install -r /content/houston_storm_layers/requirements.txt


## 2. Train and evaluate the household-burden model

449,932 raw FEMA registration records are summarized by ZIP. They produce **151 matched ZIP training/evaluation examples**, not 449,932 independent ZIP examples. The target is summed reported household members in registration records marked with flood damage. These are not verified distinct affected residents or people requesting a particular service.

Inputs are already-reported total registrations and location. Training and validation both use Harvey. Holdout: 113 training ZIPs, 38 test ZIPs. Three-fold cross-fitting supplies map estimates without fitting each ZIP's own outcome. Nearby ZIPs may appear in both subsets; this is not validation on another storm or spatially separate region.

The reference run gives baseline MAE approximately **1,151 reported members per ZIP**, versus **1,147** for the forest: only **0.35% lower**. The model barely improves on the baseline and has large local error. Treat it as experimental. Report your run's actual results, not percentage accuracy.


In [ ]:
%run /content/houston_storm_layers/train_people_model.py


In [ ]:
import json
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown
project = Path('/content/houston_storm_layers')
report = json.loads((project / 'people_model_report.json').read_text())
display(pd.DataFrame([
    {'Method':'Pooled training-rate baseline','MAE: reported members per ZIP':report['baseline_mae_reported_members_per_zip']},
    {'Method':'Random forest','MAE: reported members per ZIP':report['model_mae_reported_members_per_zip']}
]).round(2))
print(f"MAE reduction: {report['mae_reduction_percent']:.2f}% — not percent accuracy.")
print('The model is a post-event estimate and does not predict injuries, ER visits or service attendance.')


## Storm maps and independent resource placement

Choose Beryl, the 2024 derecho, or Harvey. The derecho is a windstorm, not a hurricane. Storm/recovery windows include possibly unrelated reports; source associations are not proof of cause.

- **Map filter:** tree removal and debris (green-yellow), recorded county power outages (red), electrical hazards, flood/drainage, building safety/code reports, or a utility snapshot with event dates. Additional cleanup-help and medical-evacuation request layers are available only for Harvey.
- **Damage evidence:** inspect source narratives mentioning water, trees, buildings/roofs or power/poles. These are keyword filters, not an engineering diagnosis or verified cause for a particular home.
- **Placement controls:** choose separate budgets for ambulance staging, repair dispatch, cooling hubs and community/medical posts. Role weights are explicit proxies, with no validated capacity or staffing assumptions. The ten-minute ambulance goal is base to affected area using modeled ordinary driving.
- **Utility snapshot refresh:** check each event's update time. The bundled feed's newest event is September 23, 2026, so it is stale at packaging. Old points stay gray. Refresh may still return old data; retrieval is not proof of current outage status.
- **Future probability:** explicitly unavailable. A history heatmap cannot be relabeled as a tested probability of future outages, tree strikes or improper infrastructure.

Included: 273,399 native county/time outage observations for nine regional counties, 48,318 selected mapped service-request records, and 246 source storm-description records. Native observation counts are not unique outages, people, or ML training-example counts. Red outages remain county-level; no neighborhood probabilities are fabricated. Unknown records and flagged count/denominator inconsistencies are gray.

Historical report weights use nearest represented area grouping, not exact ZIP membership. Use the role layer checkboxes for co-located sites, the ZIP inspector and the export button. All code/data are cached; only the optional utility refresh makes a new data request.


In [ ]:
import sys
project = Path('/content/houston_storm_layers')
if str(project) not in sys.path:
    sys.path.insert(0,str(project))
%run /content/houston_storm_layers/storm_layers.py


## Optional original household/supply planner
The following GUI uses the original Harvey household model and explicit participation assumptions. It does not estimate power outages or tree failures.


## 3. Open your planner GUI

The map covers **189 populated ZIP-area representative points**. The model matches 151; missing estimates remain missing. There are **14 candidate centers**, 12 passing an illustrative SFHA point rule. The search exhaustively compares eligible combinations to maximize selected priority weight within ten modeled driving minutes.

- **Response/medical posts:** temporary bases, with base-to-area driving.
- **Cooling/community hubs:** separate site selection, with area-to-hub driving and an additional illustrative ten-minute access target.
- **Siting priority:** experimental model counts, historical household-member sums, published HHS medical-equipment counts, or 2020 population context. Different weights have different denominators.
- **Participation, scale and supply days:** assumptions you control. Service groups overlap; do not add their totals as unique people. Default percentages are illustrative, not learned injury or utilization rates. They are not confidence intervals.
- **Unavailable candidate:** simulate a center closure; actual disaster closures, staffing and building safety are not established.

Only **two verified Harris Health ER examples** are supplied for fixed-destination driving comparisons. The nearest of these two is not necessarily the closest ER in Houston or a clinically suitable destination. Candidate hubs have no verified activation or capacity. Ordinary driving calculations do not guarantee ten-minute emergency response.

Use **Inspect your ZIP** to view one area. The map legend, 25-row preview, assigned hub demand and **Download all ZIP results** button appear below the controls. The download includes all 189 rows and scenario assumptions.


In [ ]:
%run /content/houston_storm_layers/expanded_planner.py


## 4. Inspect the additional injury/weather history

Included: **13,657 Texas event records** from 2017, 2019 and 2024, the Harvey, Imelda and Beryl years. These are all reported event types in those selected years, not just those hurricanes. County records and forecast-zone records are different geographic units; they are not assigned to ZIPs as injury labels. Reported injuries do not equal ER arrivals or future healthcare demand. A missing report is not evidence that no harm occurred.


In [ ]:
history = pd.read_csv(project / 'data/texas_storm_events_2017_2019_2024.csv', low_memory=False)
display(history.groupby(['YEAR','CZ_TYPE']).agg(
    event_records=('EVENT_ID','size'),
    reported_direct_injury_entries=('INJURIES_DIRECT','sum'),
    reported_indirect_injury_entries=('INJURIES_INDIRECT','sum')
))
print('C = county record; Z = forecast-zone record. These are event-report totals, not deduplicated patients.')


## 5. Explain and submit

Use the script below to explain the model, the planner and their limitations. Add your name, run the notebook, and prepare an accessible notebook/code link and a live demo. The supplied detailed Track 1 packet permits statistical/ML approaches and asks for model building and training; this notebook trains a numerical model. NLP/LLM fine-tuning is not required for that detailed track.

The packet requests a live demo; the supplied Kaggle text lists a 4:30 PM Central deadline. Check any organizer announcements.


In [ ]:
display(Markdown((project / 'PRESENTATION.md').read_text()))


## 6. Sources, data units and assumptions
The source guide explains the GitHub search, source URLs, matching rules and excluded data.


In [ ]:
display(Markdown((project / 'data/SOURCES.md').read_text()))


## New storm-layer sources and limitations


In [ ]:
display(Markdown((project / 'data/STORM_LAYER_SOURCES.md').read_text()))
